# JN6a - Capstone, part 1: interrogate the oracle + generate your APR

**Curriculum notebook 6a of 6.** You have reconstructed a housing record from raw permits (JN1-JN5). Now the payoff begins: produce a real **HCD-form APR**, and prepare to score it against the city's own submitted report. This half does two things - **interrogate the oracle** (the city's official report - a skill in itself) and **generate your APR** in HCD's shape.

> Clonable + read-only on the source data; it writes one *output* spreadsheet (a rendering).

## (run first) Colab setup

Fetches the data + shared modules from R2. **No-op if you already have the repo locally** (it detects a checkout and skips). On Colab / a bare session it recreates the minimal repo layout under the working directory so the config cell below finds everything unchanged.

In [1]:
# === COLAB BOOTSTRAP - fetch curriculum data + modules from R2 (NO-OP if the repo is local) ===
from pathlib import Path
import sys, urllib.request, urllib.parse, tarfile, subprocess

R2 = 'https://pub-2cee87f70da64080ab70ee0a34b55099.r2.dev/curriculum'
USE_CLEAN = False   # False: raw .xlsx path (JN1's messy-data lesson).  True (skip-ingest): permits_clean.*

_here = Path.cwd()
_have_repo = (_here/'scripts'/'build_v2').exists() or any((p/'scripts'/'build_v2').exists() for p in _here.parents)

def _get(url):
    # r2.dev sits behind Cloudflare, which 403s the default 'Python-urllib' User-Agent; send a browser UA.
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read()

if _have_repo:
    print('local repo detected - no fetch needed')
else:
    try:
        import pyarrow  # the parquet / USE_CLEAN path needs it; Colab has pandas, maybe not pyarrow
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'], check=True)
    def _fetch(url, dest):
        dest = Path(dest)
        if dest.exists():
            return                                   # cached: re-runs don't re-download
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(_get(url)); print('fetched', dest.name)
    # 1) shared modules -> ./scripts/...  (the config-cell repo-root walk then finds scripts/build_v2)
    if not (_here/'scripts'/'build_v2').exists():
        Path('modules.tgz').write_bytes(_get(f'{R2}/curriculum_modules.tar.gz'))
        _tar = tarfile.open('modules.tgz')
        try: _tar.extractall(_here, filter='data')      # py3.12+: safe extract, no deprecation warning
        except TypeError: _tar.extractall(_here)         # older python has no filter arg
        _tar.close(); Path('modules.tgz').unlink(missing_ok=True)   # tidy: drop the intermediate tarball
        print('extracted modules -> ./scripts/')
    # 2) data -> the SAME relative paths the notebooks use (raw .xlsx AND clean exports, both fetched)
    for rel in ['data/raw/cpra-downloads/BP_Annual Permit Report-2018-2022.xlsx',
                'data/raw/cpra-downloads/BP_Annual Permit Report-2023-2025.xlsx',
                'databases/hcd_apr_mirror_2026-06-17_fresh.db',
                'databases/hcd_apr_mirror.db',
                'data/processed/permits_clean.csv',
                'data/processed/permits_clean.parquet',
                'data/processed/permits_clean_README.md']:
        _fetch(f"{R2}/data/{urllib.parse.quote(rel.split('/')[-1])}", _here/rel)   # quote -> %20 for the spaced .xlsx names
    print('curriculum bundle ready (fetched from R2)')


local repo detected - no fetch needed


## Config

In [2]:
# === CONFIG - point this at YOUR city's data (clonable) ===
from pathlib import Path
import sys, glob
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'scripts' / 'build_v2').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
PERMIT_GLOB = str(REPO_ROOT / 'data/raw/cpra-downloads/BP_Annual Permit Report-*.xlsx')
ORACLE_DB   = str(REPO_ROOT / 'databases/hcd_apr_mirror_2026-06-17_fresh.db')  # the FRESH (deduped) CKAN pull
STALE_DB    = str(REPO_ROOT / 'databases/hcd_apr_mirror.db')                   # an older pull, for the currency lesson
OUTDIR      = REPO_ROOT / 'notebooks/curriculum/output'; OUTDIR.mkdir(exist_ok=True)
HEADER_ROW  = 7
sys.path.insert(0, str(REPO_ROOT / 'scripts')); sys.path.insert(0, str(REPO_ROOT / 'scripts' / 'build_v2'))
print('repo root:', REPO_ROOT)


repo root: /Users/johngage/berkeley-data


# PHASE 6a - Interrogate the oracle

The city files an **Annual Progress Report (APR)** with the state (HCD), published on the CKAN open-data portal. That is your **oracle** - the number to score against. But finding it and *trusting* it is a skill: you must locate the right tables, know what is missing, and **verify the oracle to ground truth before believing it.** An oracle is a hypothesis too.

### What's in the city's submission

We pulled Berkeley's APR from CKAN (`data.ca.gov`) into a local mirror. For another city you would pull its submission the same way. The tables: **Table A** (applications), **Table A2** (the building-activity report - entitlements, BP, and **CO/completions**). Note what is **absent**:

In [3]:
import sqlite3
con = sqlite3.connect(f'file:{ORACLE_DB}?mode=ro', uri=True)
tables = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table' AND name LIKE 'table_%'")]
print('oracle tables:', tables)
print('Table B (RHNA progress) present? ', 'table_b' in tables,
      ' <- ABSENT: cumulative RHNA progress is NOT in the mirror -> a coverage/soft lesson for later')

oracle tables: ['table_a2', 'table_a']
Table B (RHNA progress) present?  False  <- ABSENT: cumulative RHNA progress is NOT in the mirror -> a coverage/soft lesson for later


### Verify the oracle to ground truth (units, not rows)

Before trusting the oracle, reconcile it to a **known anchor**. Berkeley publicly reported **708 CO units for CY2024**. Completions in Table A2 are **units** = the sum of the 11 `CO_*_INCOME` columns - **not** a row count (a building can be one row with many units). If our sum hits 708, the oracle reads correctly.

In [4]:
import sqlite3
CO_COLS = ['CO_ACUTELY_LOW_INCOME_DR','CO_ACUTELY_LOW_INCOME_NDR','CO_EXTREMELY_LOW_INCOME_DR',
           'CO_EXTREMELY_INCOME_NDR','CO_VLOW_INCOME_DR','CO_VLOW_INCOME_NDR','CO_LOW_INCOME_DR',
           'CO_LOW_INCOME_NDR','CO_MOD_INCOME_DR','CO_MOD_INCOME_NDR','CO_ABOVE_MOD_INCOME']
CO_SUM = '+'.join(f"CAST(NULLIF({c},'') AS INT)" for c in CO_COLS)
def oracle_co_units_by_year(db):
    c = sqlite3.connect(f'file:{db}?mode=ro', uri=True)
    return dict((int(y), u or 0) for y, u in
                c.execute(f"SELECT YEAR, SUM({CO_SUM}) FROM table_a2 WHERE CO_ISSUE_DT1 <> '' GROUP BY YEAR"))
fresh = oracle_co_units_by_year(ORACLE_DB)
print('oracle CO UNITS by year (fresh):', dict(sorted(fresh.items())))
print(f'  CY2024 = {fresh[2024]}  (known anchor: 708)  ->', 'RECONCILES' if fresh[2024] == 708 else 'MISMATCH')

oracle CO UNITS by year (fresh): {2018: 229, 2019: 313, 2020: 405, 2021: 331, 2022: 828, 2023: 716, 2024: 708, 2025: 492}
  CY2024 = 708  (known anchor: 708)  -> RECONCILES


### Check currency - the city's own double-submission

An oracle can be **stale**. Berkeley accidentally submitted CY2025 **twice**, and an older mirror carries the doubled rows. Compare an older pull to the fresh one: the fresh (HCD-deduped) CY2025 is **half** the stale one. The lesson cuts both ways - *we caught the city's error*, and HCD's dedup confirms it - but only because we **verified currency** instead of trusting the first file we found.

In [5]:
stale = oracle_co_units_by_year(STALE_DB)
print(f'CY2025  stale mirror = {stale[2025]}   fresh (deduped) = {fresh[2025]}')
print(f'  the {stale[2025] - fresh[2025]} phantom units = the city double-submission (HCD later deduped it)')

CY2025  stale mirror = 984   fresh (deduped) = 492
  the 492 phantom units = the city double-submission (HCD later deduped it)


### Checkpoint 6a

In [6]:
assert fresh[2024] == 708, fresh[2024]
assert fresh[2025] == 492 and stale[2025] == 984
print('CHECKPOINT 6a PASS')
print(f'  oracle verified: CY2024 = 708 units (reconciles to anchor)')
print(f'  currency verified: CY2025 fresh = 492 (deduped) vs stale = 984 (the caught double-submission)')

CHECKPOINT 6a PASS
  oracle verified: CY2024 = 708 units (reconciles to anchor)
  currency verified: CY2025 fresh = 492 (deduped) vs stale = 984 (the caught double-submission)


# PHASE 6b - Generate your APR in HCD's form

Now render *your* reconstructed completions (JN1-JN5) into HCD's **Table A2** shape: completion **units by reporting year**, with the income-tier breakdown **where you have it**. You don't - the permit feed carries almost no affordability tiers - so those cells are **explicitly blank/flagged**. That is the honest-partial principle: **the spreadsheet is a rendering of your data, and the blanks are a finding** (affordability is largely invisible in the built-permit record), not something to fabricate.

In [7]:
import pandas as pd
from collections import defaultdict
from housing_predicates import is_housing, net_units
from s0_keys import normalize_address
from cpra_dedup import extract_master_permit
def pdate(x):
    d = pd.to_datetime(str(x), errors='coerce'); return d.date() if pd.notna(d) else None
def load(p):
    d = pd.read_excel(p, dtype=str, header=HEADER_ROW); d.columns = [str(c).strip() for c in d.columns]; return d
df = pd.concat([load(f) for f in glob.glob(PERMIT_GLOB)], ignore_index=True)
df = df[df['PermitNumber'].notna()].rename(columns={'Finaled Date':'FinaledDate'}).copy()
df['isnew'] = df['Work Type'].astype(str).str.strip() == 'New'
df = df[[is_housing(o,u,n,a) for o,u,n,a in zip(df['OccType'],df['UnitsAdded'],df['NumberUnits'],df['ADU'])]]
bld = defaultdict(lambda: {'units': 0.0, 'hasnew': False, 'final': []})
for r in df.itertuples(index=False):
    st = r.StreetType; st = '' if (st is None or str(st).strip().lower() == 'nan') else str(st)
    k = normalize_address(f'{r.StreetNumber} {r.StreetName} {st}'.strip())
    if not k.number: continue
    b = bld[(k.number, k.street, k.stype)]
    b['units'] = max(b['units'], net_units(r.isnew, r.UnitsAdded, r.NumberUnits, r.ADU))
    if r.isnew: b['hasnew'] = True
    pn = str(r.PermitNumber)
    if extract_master_permit(pn) == pn and net_units(r.isnew, r.UnitsAdded, r.NumberUnits, r.ADU) > 0:
        f = pdate(r.FinaledDate)
        if f: b['final'].append(f)
completions = {k: b for k, b in bld.items() if (b['hasnew'] or b['units'] > 0) and b['final']}
my_units = defaultdict(int)
for b in completions.values(): my_units[max(b['final']).year] += int(b['units'])
print('MY APR - completion units by reporting_year:', dict(sorted(my_units.items())), 'sum', sum(my_units.values()))

MY APR - completion units by reporting_year: {2018: 228, 2019: 309, 2020: 398, 2021: 368, 2022: 679, 2023: 845, 2024: 783, 2025: 700} sum 4310


### Render the Table A2 shape - honest-partial

Total completion units per year we **know** (sourced). The affordability tier split (VLI/LI/MOD/...) we **don't** - so we write the total and mark the tiers `UNKNOWN (not in permit feed)`. We then save it to an `.xlsx` - a *rendering* of the reconstructed data.

In [8]:
import openpyxl
rows = []
for y in range(2018, 2026):
    rows.append({'YEAR': y, 'CO_TOTAL_UNITS': my_units[y],
                 'CO_VLOW': 'UNKNOWN', 'CO_LOW': 'UNKNOWN', 'CO_MOD': 'UNKNOWN',
                 'CO_ABOVE_MOD': 'UNKNOWN', 'NOTE': 'tier split not in permit feed (transparency gap)'})
table_a2 = pd.DataFrame(rows)
out_path = OUTDIR / 'my_apr_table_a2.xlsx'
table_a2.to_excel(out_path, index=False, sheet_name='Table A2 (completions)')
print('wrote', out_path)
table_a2[['YEAR','CO_TOTAL_UNITS','CO_VLOW','CO_ABOVE_MOD','NOTE']]

wrote /Users/johngage/berkeley-data/notebooks/curriculum/output/my_apr_table_a2.xlsx


,YEAR,CO_TOTAL_UNITS,CO_VLOW,CO_ABOVE_MOD,NOTE
0,2018,228,UNKNOWN,UNKNOWN,tier split not in permit feed (transparency gap)
1,2019,309,UNKNOWN,UNKNOWN,tier split not in permit feed (transparency gap)
2,2020,398,UNKNOWN,UNKNOWN,tier split not in permit feed (transparency gap)
3,2021,368,UNKNOWN,UNKNOWN,tier split not in permit feed (transparency gap)
4,2022,679,UNKNOWN,UNKNOWN,tier split not in permit feed (transparency gap)
5,2023,845,UNKNOWN,UNKNOWN,tier split not in permit feed (transparency gap)
6,2024,783,UNKNOWN,UNKNOWN,tier split not in permit feed (transparency gap)
7,2025,700,UNKNOWN,UNKNOWN,tier split not in permit feed (transparency gap)


### Checkpoint 6b

In [9]:
assert sum(my_units.values()) == 4310            # the units the JN-chain reconstructed
assert my_units[2024] == 783
assert (OUTDIR / 'my_apr_table_a2.xlsx').exists()
assert (table_a2['CO_VLOW'] == 'UNKNOWN').all()  # tiers honestly flagged, never fabricated
print('CHECKPOINT 6b PASS')
print(f'  my APR rendered: {sum(my_units.values())} total completion units (CY2024 = {my_units[2024]})')
print('  HCD-form xlsx written; affordability tiers explicitly flagged UNKNOWN (honest blanks)')

CHECKPOINT 6b PASS
  my APR rendered: 4310 total completion units (CY2024 = 783)
  HCD-form xlsx written; affordability tiers explicitly flagged UNKNOWN (honest blanks)


**JN6a done.** You have a verified oracle and your own HCD-form APR. **Next - JN6b:** join the two, decompose the difference, and let your own scorecard rediscover the limitation you planted in JN3.